#### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Neural Netowrk specifc imports
import copy # for deepcopy in save_results()

import torch
import torch.nn as nn # NN layers and loss functions
import torch.optim as optim # Optimization Algorithms
# Batching Data:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

#### CONFIGURATOINS

In [2]:
FEATURE_ROOT = "speaker_wise-eGeMAPs/functionals"
OUTPUT_ROOT = "model_outputs/neural-network"
RANDOM_SEED = 42
THRESHOLD = 0.5
# TRAIN_RATIO = 0.8
# VAL_RATIO = 0.1
# TEST_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHILD_SPEAKER = {
     
    "speaker_functional_p5-s2.csv": "SPEAKER_00",
    "speaker_functional_p5-s7.csv": "SPEAKER_01",
    "speaker_functional_p5-s8.csv": "SPEAKER_05",
    "speaker_functional_p5-s10.csv": "SPEAKER_01",
    "speaker_functional_p5-s13.csv": "SPEAKER_01",

    "speaker_functional_p7-s5.csv": "SPEAKER_01",
    "speaker_functional_p7-s6.csv": "SPEAKER_08",
    "speaker_functional_p7-s7.csv": "SPEAKER_02",
    "speaker_functional_p7-s8.csv": "SPEAKER_00",
    "speaker_functional_p7-s16.csv": "SPEAKER_00",
    "speaker_functional_p7-s17.csv": "SPEAKER_00",
    "speaker_functional_p7-s18.csv": "SPEAKER_02",
    "speaker_functional_p7-s29.csv": "SPEAKER_00",

    "speaker_functional_p9-s3-1.csv": "SPEAKER_01",
    "speaker_functional_p9-s3-2.csv": "SPEAKER_04",
    "speaker_functional_p9-s4.csv": "SPEAKER_06",
    "speaker_functional_p9-s9.csv": "SPEAKER_03",
    "speaker_functional_p9-s15.csv": "SPEAKER_01",

    "speaker_functional_p11-s2.csv": "SPEAKER_02",
    "speaker_functional_p11-s4.csv": "SPEAKER_01",
    "speaker_functional_p11-s8.csv": "SPEAKER_03",
    "speaker_functional_p11-s9.csv": "SPEAKER_00",
    "speaker_functional_p11-s11.csv": "SPEAKER_05",
    "speaker_functional_p11-s15.csv": "SPEAKER_00",
    "speaker_functional_p11-s16-2.csv": "SPEAKER_01",
    "speaker_functional_p11-s19.csv": "SPEAKER_00",
    "speaker_functional_p11-s22-2.csv": "SPEAKER_02",

    "speaker_functional_p12-s2-2.csv": "SPEAKER_00",
    "speaker_functional_p12-s3.csv": "SPEAKER_01",
    "speaker_functional_p12-s6.csv": "SPEAKER_01",
    "speaker_functional_p12-s8.csv": "SPEAKER_00",
    "speaker_functional_p12-s10.csv": "SPEAKER_03",

    "speaker_functional_p17-s2.csv": "SPEAKER_01",
    "speaker_functional_p17-s3.csv": "SPEAKER_04",
    "speaker_functional_p17-s5.csv": "SPEAKER_04",
    "speaker_functional_p17-s6.csv": "SPEAKER_02",

    "speaker_functional_p18-s3.csv": "SPEAKER_00",
    "speaker_functional_p18-s4.csv": "SPEAKER_01",
    "speaker_functional_p18-s5.csv": "SPEAKER_00",
    "speaker_functional_p18-s7.csv": "SPEAKER_01",
    "speaker_functional_p18-s8.csv": "SPEAKER_00",
    "speaker_functional_p18-s9.csv": "SPEAKER_01",
    "speaker_functional_p18-s10.csv": "SPEAKER_06",
    "speaker_functional_p18-s11.csv": "SPEAKER_00",
    "speaker_functional_p18-s12.csv": "SPEAKER_00",
    "speaker_functional_p18-s13.csv": "SPEAKER_01",
    "speaker_functional_p18-s15.csv": "SPEAKER_02",
    "speaker_functional_p18-s17.csv": "SPEAKER_01",
    "speaker_functional_p18-s18.csv": "SPEAKER_00",
    "speaker_functional_p18-s19.csv": "SPEAKER_02",
    "speaker_functional_p18-s20.csv": "SPEAKER_01",

}

#### FUNCTIONS

In [3]:
# Early stopping
PATIENCE = 10 # Stop after 10 epochs of no improvement in validation loss

In [4]:
def load_participant_data(participant_folder):
    # Load all CSV files for a given participant folder in a sorted fashion
    csv_files = sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv")) # For multiple files

    #  Function to filter the child speaker from single csv file
    def load_single_speaker(csv_path):
        df = pd.read_csv(csv_path)    
        filename = os.path.basename(csv_path)

        # Error handle
        if filename not in CHILD_SPEAKER:
            raise ValueError(
                f"No child speaker mapping for {filename}"
            )
        
        target_speaker = CHILD_SPEAKER[filename]
        
        # df = df[True] where it is True for child speaker
        df = df[df["speaker"] == target_speaker]

        return df

    ### DEBUG STATEMENT   
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    ###

    # Load every session for this participant
    full_df = pd.concat(
        [
            # Load child speaker only from file f
            load_single_speaker(f)
            for f in csv_files
        ],
        ignore_index=True
    )
     
    ### DEBUG STATEMENT
    print(f"Total child utterances: {len(full_df)}")
    ### 

    # Remove unnecessary columns from eGeMAPs table
    drop_cols = [
        "participant",
        "session",
        "clip_id",
        "speaker",

        "engagement_start_time",
        "engagement_end_time",

        "speaker_start_time",
        "speaker_end_time",

        "num_segments",
        "speech_duration",
    ]

    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )

    print(full_df.columns)

    # Separate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    # We reset the index below because we'ev filtered non-child speaker rows
    return (
        X.reset_index(drop=True),
        y.reset_index(drop=True)
    )

In [5]:
def preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_val = encoder.transform(y_val)
    y_test = encoder.transform(y_test)

    ### DEBUG STATEMENT
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))
    ###
    
    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_val = scaler.transform(X_val) # Transform the data using the learned mean and std
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        scaler,
        encoder
    )  

In [6]:
class NeuralNetwork(nn.Module):

    # Define the architecture of the neural network ; Constructor
    def __init__(self, input_dim):

        super().__init__() # Initialize the parent class (nn.Module) first, then inherit functionalities

        self.network = nn.Sequential(
            # First Hidden Layer 88 -> 128 nueruons
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Second Hidden Layer 128 -> 64 nueruons
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output Layer 64 -> 1 nuerons
            nn.Linear(64, 1)

        )
    
    # Forward pass through the network; Automatically called  when calling model; Then return the model predictions
    def forward(self, x):
        return self.network(x)


# Build the model, then move it to GPU/ CPU and print the model architecture
def build_model(input_dim):
    # Build model
    model = NeuralNetwork(input_dim)
    
    # Move model to GPU/ CPU
    model.to(DEVICE)
    
    print(model)

    return model

In [7]:
def train_model(
        model,
        train_loader,
        val_loader
):
    # BCEWithLogitsLoss is good for our case of binary classification
    # It performs sigmoid + Binary Cross Entropy Loss 
    criterion = nn.BCEWithLogitsLoss()

    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Initialize best loss to be infinity and patience counter to 0
    best_loss = float("inf")
    patience_counter = 0

    # Store training and validation loss for each epoch (perhaps for plotting later)
    history = {
        "train_loss": [],
        "val_loss": []
    }

    # Epoch Loop; Max is 100, but may stop earlier due to early stopping
    for epoch in range(100):
        
        # TRAINING STEPS:
        # Activate training mode (Dropout layers and gradient computation)
        model.train()
        train_loss = 0

        # Train Batch-wise (32)
        for X_batch, y_batch in train_loader:
            
            # Move batch to GPU/ CPU
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            # Reset the gradients before backpropagation
            optimizer.zero_grad()

            # Foeward pass prediction
            outputs = model(X_batch)

            # Compute loss
            loss = criterion(outputs, y_batch)

            # Backpropagation 
            # Computes gradients
            loss.backward()
            # Update weights using optimizer
            optimizer.step()
            # Accumute training loss
            train_loss += loss.item() # loss is a tensor

        # Average traingin loss per batch
        train_loss /= len(train_loader)

        # VALIDATION STEPS
        # Turn off training mode( No Dropout layers and gradient computation)
        model.eval()
        val_loss = 0

        # Disable gradient computation
        with torch.no_grad():
            # Validate Batch-wise (32)
            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
 
                # Calls the forward pass of the model to get predictions
                outputs = model(X_batch)
                # Compute validation loss; criterion is the loss function defined in above 
                loss = criterion(outputs, y_batch)
                #Accumulate validation loss
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"Train={train_loss:.4f} "
            f"Val={val_loss:.4f}"
        )

        # If validation loss improves, save the model weights and reset patience counter
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

        # If validation loss does not improve, increment patience counter and check for early stopping
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping")
                # Exit training loop
                break

    # Restore the best model weights after training is complete
    model.load_state_dict(best_weights)

    return history

In [8]:
def evaluate_model(
        model,
        test_loader,
        encoder
):
    # Turn off training mode( No Dropout layers and gradient computation)
    model.eval()

    probabilities = [] # Store the predicted probabilities for the positive class (engaged)
    predictions = [] # Store the predicted class labels (0 or 1)
    actual = [] # Store the actual class labels (0 or 1)

    # Disable gradient computation
    with torch.no_grad():
        # Exaluate Batch-wise (32)
        for X_batch, y_batch in test_loader:
            
            X_batch = X_batch.to(DEVICE)
            # Forward pass prediction
            outputs = model(X_batch)
            # Apply sigmoid to get probabilities (0 to 1)
            probs = torch.sigmoid(outputs)
            # Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
            preds = (probs >= THRESHOLD).float()

            # NumPy cannotread GPU tensors, so we need to move them to CPU and convert to NumPy arrays before storing them in lists
            probabilities.extend(probs.cpu().numpy().flatten())
            predictions.extend(preds.cpu().numpy().flatten())
            actual.extend(y_batch.numpy().flatten())

    # Convert lists to NumPy arrays and ensure they are of integer type
    probabilities = np.array(probabilities).astype(float)
    predictions = np.array(predictions).astype(int)
    actual = np.array(actual).astype(int)

    # DEBUGGING STATEMENTS
    # How many samples are actually there from each class
    print("\nActual class counts:")
    print(pd.Series(actual).value_counts()) 
    # How many samples were predicted from each class
    print("\nPredicted class counts:")
    print(pd.Series(predictions).value_counts())
    # Print first 20 probabilty values
    print("\nPredicted probabilities:")
    print(probabilities[:20])

    print(f"\nMinimum probability : {probabilities.min():.3f}")
    print(f"Maximum probability : {probabilities.max():.3f}")
    print(f"Average probability : {probabilities.mean():.3f}")

    # actual contains the true labels; filter the engaged and disengaged probabilities based on the actual labels
    engaged_probs = probabilities[actual == 1]
    disengaged_probs = probabilities[actual == 0]

    print("\nProbability Statistics")
    print("----------------------")
    print(f"Engaged mean      : {engaged_probs.mean():.3f}")
    print(f"Engaged std       : {engaged_probs.std():.3f}")
    print(f"Disengaged mean   : {disengaged_probs.mean():.3f}")
    print(f"Disengaged std    : {disengaged_probs.std():.3f}")

    # plt.figure(figsize=(6,4))
    # plt.hist(engaged_probs, bins=15, alpha=0.6, label="Engaged")
    # plt.hist(disengaged_probs, bins=15, alpha=0.6, label="Disengaged")
    # plt.xlabel("Predicted Probability")
    # plt.ylabel("Count")
    # plt.legend()
    # plt.show()

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions)
    }

    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True # Return the report as a dictionary instead of a string to make csv
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [9]:
''' The following are saved:
    Model weights (model.pt)
    Scaler (scaler.pkl)
    Encoder (encoder.pkl)
    Training History (history.csv)
    Metrics (metrics.csv)
    Confusion Matrix (confusion_matrix.csv)
    Classification Report (classification_report.csv) '''
    
def save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    ):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    torch.save(model.state_dict(),participant_output / "neural-network.pt")
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save History
    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(participant_output, "history.csv"),index=False)
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [10]:
summary_results = [] # Store average metircs for each participant
participants = sorted(os.listdir(FEATURE_ROOT))
all_fold_results = [] # Store metrics for each fold of each participant

# Loop through each participant and train a model for each participant
for participant in participants:

    print(f"Training {participant}")
    # Load ALL data for this participant; X= GeMAPs features, y= labels (engaged/disengaged)
    X, y = load_participant_data(participant)

    skf = StratifiedKFold(
        n_splits=5, # 5 folds
        shuffle=True, # randomise the samples before splirtting
        random_state=RANDOM_SEED
    )
    # Store metrics for each fold of this participant
    participant_metrics = []

    # Loop through each fold; train_idx= indices for training samples, test_idx= indices for testing samples
    # 80% train, 20% test;
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        print(f"\nFold {fold}/5")
        # Create Train/Test feature set based on train indexes for current fold
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        # Create Train/Test label set based on train indexes for current fold
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Split again: 80% of train (=64%) used for training, 20% of train (=16%) used for validation
        X_train, X_val, y_train, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            stratify=y_train, # Maintain class distribution
            random_state=RANDOM_SEED
        )

        # DEBUGGING STATEMENTS
        print("\nTraining class distribution:")
        print(y_train.value_counts())

        print("\nValidation class distribution:")
        print(y_val.value_counts())

        print("\nTesting class distribution:")
        print(y_test.value_counts())
        ###

        # Preprocess data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler, encoder = preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test)

        # Convert NumPy arrays into PyTorch tensors.
        X_train_tensor = torch.FloatTensor(X_train)
        X_val_tensor = torch.FloatTensor(X_val)
        X_test_tensor = torch.FloatTensor(X_test)

        # Shape of y_train, y_val, y_test is (N,), but we need (N,1) for BCEWithLogitsLoss
        y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
        y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)
        y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

        # Create TensorDatasets; Join features with corresponding labels
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

        # Create DataLoaders; Batch the data (32) and randomly shuffle
        # FOr trainging dataset
        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True # Shuffle training data for better generalization
        )
        # FOr validation dataset
        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False
        )
        # FOr testing dataset
        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=False
        )

        # Build model
        model = build_model(X_train.shape[1])

        # Train model and return training history
        history = train_model(model, train_loader, val_loader)

        # Evaluate model and return evaluation metrics
        evaluation = evaluate_model(model, test_loader, encoder)

        # Append metrics for this participant to the summary results
        participant_metrics.append(evaluation["metrics"])

        # Append fold results to all_fold_results
        all_fold_results.append({
            "Participant": participant,
            "Fold": fold,
            # evaluation["metrics"] is a dictionary containing the metrics for this fold, ** unpacks the dictionary 
            **evaluation["metrics"]
        })
        
        # Save results
        save_results(
            f"{participant}/fold_{fold}",
            model,
            scaler,
            encoder,
            history,
            evaluation
        )
    
    metrics_df = pd.DataFrame(participant_metrics)
    
    summary_results.append({
        "Participant": participant,

        "Accuracy Mean": metrics_df["Accuracy"].mean(),
        "Accuracy Std": metrics_df["Accuracy"].std(),

        "Precision Mean": metrics_df["Precision"].mean(),
        "Precision Std": metrics_df["Precision"].std(),

        "Recall Mean": metrics_df["Recall"].mean(),
        "Recall Std": metrics_df["Recall"].std(),

        "F1 Mean": metrics_df["F1 Score"].mean(),
        "F1 Std": metrics_df["F1 Score"].std()
    })
      
    pd.DataFrame(all_fold_results).to_csv(
        "fold_results.csv",
        index=False
    )

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    

Training p11
p11: 9 CSV files
Total child utterances: 272
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', '

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 8 Train=0.5496 Val=0.6089
Epoch 9 Train=0.5428 Val=0.5981
Epoch 10 Train=0.5429 Val=0.5884
Epoch 11 Train=0.5069 Val=0.5686
Epoch 12 Train=0.4872 Val=0.5542
Epoch 13 Train=0.4619 Val=0.5449
Epoch 14 Train=0.4536 Val=0.5406
Epoch 15 Train=0.4327 Val=0.5381
Epoch 16 Train=0.4022 Val=0.5361
Epoch 17 Train=0.4012 Val=0.5218
Epoch 18 Train=0.3882 Val=0.5234
Epoch 19 Train=0.3854 Val=0.5161
Epoch 20 Train=0.3373 Val=0.5320
Epoch 21 Train=0.3274 Val=0.5458
Epoch 22 Train=0.2707 Val=0.5521
Epoch 23 Train=0.2688 Val=0.5797
Epoch 24 Train=0.2515 Val=0.5977
Epoch 25 Train=0.2565 Val=0.6048
Epoch 26 Train=0.2320 Val=0.5809
Epoch 27 Train=0.2251 Val=0.6005
Epoch 28 Train=0.1903 Val=0.6393
Epoch 29 Train=0.1907 Val=0.6482
Early stopping

Actual class counts:
1    35
0    19
Name: count, dtype: int64

Predicted class counts:
1    42
0    12
Name: count, dtype: int64

Predicted probabilities:
[0.84839761 0.90320367 0.14590277 0.46023098 0.31630906 0.76604885
 0.93557054 0.90299904 0.645365   0.8

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 18 Train=0.2953 Val=0.7704
Epoch 19 Train=0.2403 Val=0.7971
Early stopping

Actual class counts:
1    34
0    18
Name: count, dtype: int64

Predicted class counts:
1    44
0     8
Name: count, dtype: int64

Predicted probabilities:
[0.69947523 0.63131416 0.8510775  0.79849893 0.74828726 0.51804119
 0.67372733 0.79088759 0.5447138  0.49667099 0.37053981 0.66606039
 0.63869971 0.77889538 0.702766   0.66027182 0.76183903 0.81804085
 0.48749095 0.82259846]

Minimum probability : 0.371
Maximum probability : 0.921
Average probability : 0.656

Probability Statistics
----------------------
Engaged mean      : 0.672
Engaged std       : 0.135
Disengaged mean   : 0.626
Disengaged std    : 0.125
              precision    recall  f1-score   support

  disengaged       0.50      0.22      0.31        18
     engaged       0.68      0.88      0.77        34

    accuracy                           0.65        52
   macro avg       0.59      0.55      0.54        52
weighted avg       0.62      

...

Epoch 2 Train=0.6541 Val=0.6603
Epoch 3 Train=0.6083 Val=0.6470
Epoch 4 Train=0.6536 Val=0.6349
Epoch 5 Train=0.5898 Val=0.6275
Epoch 6 Train=0.6217 Val=0.6190
Epoch 7 Train=0.6004 Val=0.6102
Epoch 8 Train=0.5952 Val=0.6032
Epoch 9 Train=0.5378 Val=0.5946
Epoch 10 Train=0.5420 Val=0.5885
Epoch 11 Train=0.4971 Val=0.5859
Epoch 12 Train=0.4957 Val=0.5852
Epoch 13 Train=0.4917 Val=0.5845
Epoch 14 Train=0.4489 Val=0.5847
Epoch 15 Train=0.4348 Val=0.5883
Epoch 16 Train=0.4071 Val=0.5928
Epoch 17 Train=0.3901 Val=0.5997
Epoch 18 Train=0.4319 Val=0.6086
Epoch 19 Train=0.4028 Val=0.6202
Epoch 20 Train=0.3507 Val=0.6321
Epoch 21 Train=0.2900 Val=0.6462
Epoch 22 Train=0.3160 Val=0.6628
Epoch 23 Train=0.3600 Val=0.6822
Early stopping

Actual class counts:
1    15
0     7
Name: count, dtype: int64

Predicted class counts:
1    22
Name: count, dtype: int64

Predicted probabilities:
[0.66789502 0.70082188 0.63554484 0.67018348 0.6787082  0.79420769
 0.66436464 0.67819774 0.84862411 0.74766588 0.7156

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

              precision    recall  f1-score   support

  disengaged       0.00      0.00      0.00         5
     engaged       0.69      1.00      0.81        11

    accuracy                           0.69        16
   macro avg       0.34      0.50      0.41        16
weighted avg       0.47      0.69      0.56        16


Fold 2/5

Training class distribution:
label
engaged       35
disengaged    16
Name: count, dtype: int64

Validation class distribution:
label
engaged       9
disengaged    4
Name: count, dtype: int64

Testing class distribution:
label
engaged       11
disengaged     5
Name: count, dtype: int64
['disengaged' 'engaged']
[0 1]
[0 1]
[0 1]
NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=88, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 7 Train=0.6353 Val=0.6095
Epoch 8 Train=0.6027 Val=0.6112
Epoch 9 Train=0.5850 Val=0.6131
Epoch 10 Train=0.5532 Val=0.6156
Epoch 11 Train=0.5693 Val=0.6197
Epoch 12 Train=0.5615 Val=0.6232
Epoch 13 Train=0.5066 Val=0.6249
Epoch 14 Train=0.4903 Val=0.6287
Epoch 15 Train=0.4730 Val=0.6351
Epoch 16 Train=0.4751 Val=0.6440
Epoch 17 Train=0.4492 Val=0.6542
Early stopping

Actual class counts:
1    9
0    7
Name: count, dtype: int64

Predicted class counts:
1    15
0     1
Name: count, dtype: int64

Predicted probabilities:
[0.64533675 0.65798545 0.54953229 0.51931751 0.63081747 0.60241842
 0.57042533 0.4791728  0.59993023 0.55923742 0.65352553 0.58727241
 0.63797605 0.62189293 0.61882359 0.59371781]

Minimum probability : 0.479
Maximum probability : 0.658
Average probability : 0.595

Probability Statistics
----------------------
Engaged mean      : 0.625
Engaged std       : 0.028
Disengaged mean   : 0.558
Disengaged std    : 0.043
              precision    recall  f1-score   support
